**데이터 전처리(Data Preprocessing) 개념 정리**

**1. 데이터 전처리 정의 및 목적**

- **정의**: 원시 데이터(raw data)를 분석 및 머신러닝에 적합한 형태 변환
- **목적**: 데이터 품질 향상, 분석/모델링 과정 오류 최소화 및 정확한 결과 도출

**2. 데이터 전처리  작업**

- **정제(Cleansing)**
    - **결측치 처리**:
        - 개념: 누락된 값(Missing values) 식별 및 삭제/대체
        - *식별*: 단순 탐색, 시각화, 결측치 패턴 분석
        - *처리*: 삭제, 평균/중앙값/최빈값 대체, 전진/후진 채우기, 임의 값, KNN/회귀/다중 대체
    - **이상치 처리**:
        - 개념 :  다른 데이터 포인트와 현저히 차이 나는 값 처리
        - *식별*: IQR 방법, 박스플롯(Boxplot) 사용, 상한선/하한선 설정
        - *처리*: 이상치 제거, 평균/중앙값/최빈값 대체, 로그/제곱근 변환, 클리핑(Clipping), 회귀 모델/KNN 대체
    - **중복 데이터 제거**: 동일 데이터 레코드 정리
- **통합(Integration)**:
    - 개념: 분석을 위한 여러 데이터원 통합
- **변환(Transformation)**
    - 데이터 분포 정상화, 범위 표준화, 이상치 영향 감소 등을 목적으로 다루기 쉬운 형식 변환
    - 척도 맞추기 작업, 스케일링, 정규화, 범주형 데이터 인코딩
- **데이터 축소(Reduction)**: 차원(변수) 축소, 샘플링
- **특징 선택 및 생성(Feature Selection & Engineering)**: 모델 성능 향상을 위한 주요 특징(변수) 선택 및 생성

Seaborn의 titanic 데이터셋과 Scikit-learn 라이브러리를 활용해 전처리 전 과정을 구현한 예시 코드입니다.

폴더가 없을 때만 새로 생성

In [ ]:
import os


def create_folder_if_not_exists(folder_path):
  # 폴더 존재 여부 확인 (검색)
  if not os.path.exists(folder_path):
    # 폴더가 없으면 생성 (상위 폴더까지 필요시 자동 생성)
    os.makedirs(folder_path)
    print(f"폴더가 존재하지 않아 새로 생성했습니다: {folder_path}")
  else:
    print(f"이미 존재하는 폴더입니다: {folder_path}")


# 사용 예시
target_folder = "./model"
create_folder_if_not_exists(target_folder)

### 1. titanic 데이터셋 (seaborn)

- **전처리:** 중복제거, IQR 기준 클리핑(이상치), StandardScaler 스케일링, PCA 차원축소

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 데이터 로드
df = sns.load_dataset("titanic")

# ==========================================
# 1. 정제 (Cleansing)
# ==========================================

# 1-1. 결측치 처리
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

# 1-2. 이상치 처리 (IQR 클리핑)
q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
lower_bound = q1 - 1.5 * iqr
df["fare"] = np.clip(df["fare"], lower_bound, upper_bound)

# 1-3. 중복 데이터 제거 및 인덱스 재정렬 (중요: 인덱스 불일치 방지)
df = df.drop_duplicates().reset_index(drop=True)


# ==========================================
# 2. 변환 (Transformation)
# ==========================================

# 2-1. 범주형 데이터 인코딩 (One-Hot Encoding)
encoder = OneHotEncoder(sparse_output=False, drop="first")
encoded_sex = encoder.fit_transform(df[["sex"]])
encoded_sex_df = pd.DataFrame(
    encoded_sex, columns=encoder.get_feature_names_out(["sex"])
)

# 2-2. 스케일링 및 정규화
scaler = StandardScaler()
df[["age_scaled", "fare_scaled"]] = scaler.fit_transform(df[["age", "fare"]])


# ==========================================
# 3. 특징 선택 및 생성 (Feature Engineering)
# ==========================================

# 파생 변수 생성
df["family_size"] = df["sibsp"] + df["parch"] + 1

# 전처리 완료된 특징 및 타겟 결합
X_features = pd.concat(
    [df[["age_scaled", "fare_scaled", "family_size"]], encoded_sex_df], axis=1
)
y = df["survived"]


# ==========================================
# 4. 데이터 축소 및 분할 (Reduction & Split)
# ==========================================

# Train / Test 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42
)

# 차원 축소 (PCA)
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("전처리 완료 데이터 크기:", X_features.shape)
print("PCA 축소 후 Train 데이터 크기:", X_train_pca.shape)


# ==========================================
# 5. 모델 학습 & 평가 (LogisticRegression)
# ==========================================

# 최신 scikit-learn 기준 L2 규제 적용 (l1_ratio=0.0)
model = LogisticRegression(solver="saga", l1_ratio=0.0, random_state=42)
model.fit(X_train_pca, y_train)
y_pred = model.predict(X_test_pca)

print("\n--- Titanic Classification Report ---")
print(classification_report(y_test, y_pred))


# ==========================================
# 6. 폴더 확인 후 모델 저장
# ==========================================

save_dir = "model"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

model_path = os.path.join(save_dir, "titanic_logistic_model.pkl")
joblib.dump(model, model_path)
print(f"모델 저장 완료: {model_path}")

전처리 완료 데이터 크기: (778, 4)
PCA 축소 후 Train 데이터 크기: (622, 2)

--- Titanic Classification Report ---
              precision    recall  f1-score   support

           0       0.58      0.94      0.72        86
           1       0.71      0.17      0.28        70

    accuracy                           0.60       156
   macro avg       0.64      0.56      0.50       156
weighted avg       0.64      0.60      0.52       156

모델 저장 완료: model\titanic_logistic_model.pkl
